# Forecasting Under Regime Instability: Results and Interpretation of Regime-Dependent Predictability

## Purpose
This notebook consolidates results across all the previous analyses to answer the question: did labor market signals stay reliable predictors after COVID-19, or did their predictive power become regime-dependent? The analysis reuses code from feature engineering, baseline models, structural benchmarks, and forecast models to create which signals retained predictive value, which fell entirely, and how model architecture affected performance in unstable enviroments.

## Imports and Configuration

In [ ]:
%matplotlib inline

from IPython.display import display, Markdown

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from matplotlib.colors import ListedColormap
from pathlib import Path

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from regime_shift.config import (
    FIG_ROOT as fig_root,
    MODEL_ROOT as model_root,
    PRICE_MODEL_READY_PATH as price_model_ready_path,
    REPORT_ROOT as report_root,
    SAVE_DPI as save_dpi,
    SHOW_DEC as show_dec,
    WAGE_MODEL_READY_PATH as wage_model_ready_path,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

for path in [fig_root, model_root, report_root]:
    Path(path).mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")

## Load Wage and Price Dataset

In [ ]:
wage_data = pd.read_csv(wage_model_ready_path)
price_data = pd.read_csv(price_model_ready_path)

for data in [wage_data, price_data]:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data.dropna(subset=["date"], inplace=True)
    data.sort_values("date", inplace=True)
    data.reset_index(drop=True, inplace=True)

## Target Construction and Model Functions

### Define Forecast Targets

In [ ]:
def target_map() -> dict:
    return {
        "wage": [
            "wage_target_3",
            "wage_target_6",
            "wage_target_12",
        ],
        "price": [
            "cpi_target_3",
            "cpi_target_6",
            "cpi_target_12",
            "pce_target_3",
            "pce_target_6",
            "pce_target_12",
        ],
    }

In [ ]:
all_targets = target_map()["wage"] + target_map()["price"]

### Define Models and Model Families

In [ ]:
model_list = [
    "Persistence",
    "AR",
    "OLS",
    "Ridge",
    "ElasticNet",
    "Huber",
    "RandomForest",
    "GradientBoosting",
]

In [ ]:
linear_list = ["OLS", "Ridge", "ElasticNet", "Huber"]
tree_list = ["RandomForest", "GradientBoosting"]
base_list = ["Persistence", "AR"]

### Define Labor Market Signals and Core Controls

In [ ]:
signal_list = [
    "jolts",
    "unemployment",
    "quits",
    "fed_funds",
    "lagged_target",
]

In [ ]:
core_controls = ["fed_funds", "hy_oas"]

### Horizon and Target Labels

In [ ]:
def horizon(target_col: str) -> int:
    return int(target_col.rsplit("_", 1)[-1])

In [ ]:
def target_name(target_col: str) -> str:
    if target_col.startswith("wage_"):
        return "Wage"
    if target_col.startswith("cpi_"):
        return "CPI"
    if target_col.startswith("pce_"):
        return "PCE"
    raise ValueError("Target column isn't supported.")

In [ ]:
def target_label(target_col: str) -> str:
    return f"{target_name(target_col)} {horizon(target_col)}m"

In [ ]:
def unique_cols(cols: list[str]) -> list[str]:
    out = []
    seen = set()

    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)

    return out

### Construct Lagged Targets and Feature Blocks

In [ ]:
def lag_col_name(target_col: str) -> str:
    return f"{target_col}_lag"

In [ ]:
def add_lag(data: pd.DataFrame, target_col: str) -> pd.DataFrame:
    out = data.copy()
    out[lag_col_name(target_col)] = out[target_col].shift(horizon(target_col))
    return out

## Forecast Evaluation Metrics

In [ ]:
def rmse_value(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(actual, pred)))


def mae_value(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(mean_absolute_error(actual, pred))


def oos_r2(actual: np.ndarray, pred: np.ndarray, base_pred: np.ndarray) -> float:
    model_mse = mean_squared_error(actual, pred)
    base_mse = mean_squared_error(actual, base_pred)

    if base_mse == 0:
        return np.nan

    return float(1.0 - model_mse / base_mse)


def min_train_rows(target_col: str, regime_name: str) -> int:
    if regime_name == "pre":
        return max(48, horizon(target_col) * 8)

    return max(8, horizon(target_col))

## Model Objects and Estimation Rules

In [ ]:
def base_block(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, list[str], str]:
    out = add_lag(data, target_col)
    lag_col = lag_col_name(target_col)

    feat_cols = [
        "log_jolts_ratio",
        lag_col,
        "fed_funds",
        "hy_oas",
    ]
    feat_cols = [col for col in feat_cols if col in out.columns]

    keep_cols = unique_cols(
        [
            "date",
            target_col,
            *feat_cols,
            "pre_regime",
            "shock_regime",
            "post_regime",
        ]
    )

    out = out.loc[:, keep_cols].copy()

    for col in out.columns:
        if col != "date":
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna().reset_index(drop=True)

    return out, feat_cols, lag_col

In [ ]:
def ar_block(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, list[str], str]:
    out = data.copy()
    ar_col = f"{target_col}_ar1"
    out[ar_col] = out[target_col].shift(1)

    keep_cols = unique_cols(
        [
            "date",
            target_col,
            ar_col,
            "pre_regime",
            "shock_regime",
            "post_regime",
        ]
    )

    out = out.loc[:, keep_cols].copy()

    for col in out.columns:
        if col != "date":
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna().reset_index(drop=True)

    return out, [ar_col], ar_col

In [ ]:
def signal_block(data: pd.DataFrame, target_col: str, signal_name: str) -> tuple[pd.DataFrame, list[str], str, str]:
    out = add_lag(data, target_col)
    lag_col = lag_col_name(target_col)

    if signal_name == "jolts":
        signal_col = "log_jolts_ratio"
        feat_cols = [signal_col, lag_col, "fed_funds", "hy_oas"]

    elif signal_name == "unemployment":
        signal_col = "unemployment_rate"
        feat_cols = [signal_col, lag_col, "fed_funds", "hy_oas"]

    elif signal_name == "quits":
        signal_col = "quits_rate"
        feat_cols = [signal_col, lag_col, "fed_funds", "hy_oas"]

    elif signal_name == "fed_funds":
        signal_col = "fed_funds"
        feat_cols = [signal_col, lag_col, "hy_oas"]

    elif signal_name == "lagged_target":
        signal_col = lag_col
        feat_cols = [signal_col]

    else:
        raise ValueError("Signal is not supported.")

    feat_cols = [col for col in feat_cols if col in out.columns]

    keep_cols = unique_cols(
        [
            "date",
            target_col,
            *feat_cols,
            "pre_regime",
            "shock_regime",
            "post_regime",
        ]
    )

    out = out.loc[:, keep_cols].copy()

    for col in out.columns:
        if col != "date":
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna().reset_index(drop=True)

    return out, feat_cols, lag_col, signal_col

### Model Maps

In [ ]:
linear_model_map = {
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.03, l1_ratio=0.50, max_iter=20000, random_state=42),
    "Huber": HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=500),
}

In [ ]:
tree_model_map = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.90,
        min_samples_leaf=5,
        random_state=42,
    ),
}

### Model Fitting

In [ ]:
def fit_persistence(train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    last_value = float(train_target.iloc[-1])
    return np.repeat(last_value, len(test_feat))


def fit_ols(train_feat: pd.DataFrame, train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    train_data = sm.add_constant(train_feat, has_constant="add")
    test_data = sm.add_constant(test_feat, has_constant="add")
    test_data = test_data.reindex(columns=train_data.columns, fill_value=0.0)

    fit = sm.OLS(train_target, train_data).fit()
    pred = fit.predict(test_data)

    return np.asarray(pred)


def fit_pipe(model, train_feat: pd.DataFrame, train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    pipe = Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", clone(model)),
        ]
    )

    pipe.fit(train_feat, train_target)
    pred = pipe.predict(test_feat)

    return np.asarray(pred)


def fit_tree(model, train_feat: pd.DataFrame, train_target: pd.Series, test_feat: pd.DataFrame) -> np.ndarray:
    fit = clone(model)
    fit.fit(train_feat, train_target)
    pred = fit.predict(test_feat)

    return np.asarray(pred)


def model_pred(
    model_name: str,
    train_feat: pd.DataFrame,
    train_target: pd.Series,
    test_feat: pd.DataFrame,
    ar_col: str,
) -> np.ndarray:
    if model_name == "Persistence":
        return fit_persistence(train_target, test_feat)

    if model_name == "AR":
        return fit_ols(train_feat[[ar_col]], train_target, test_feat[[ar_col]])

    if model_name == "OLS":
        return fit_ols(train_feat, train_target, test_feat)

    if model_name in linear_model_map:
        return fit_pipe(linear_model_map[model_name], train_feat, train_target, test_feat)

    if model_name in tree_model_map:
        return fit_tree(tree_model_map[model_name], train_feat, train_target, test_feat)

    raise ValueError("Model isn't supported.")

## Pre-COVID and Post-COVID Validation Splits

In [ ]:
def pre_points(data: pd.DataFrame, target_col: str, folds: int = 5) -> list[tuple[slice, slice]]:
    row_count = len(data)
    min_train = min_train_rows(target_col, "pre")

    if row_count <= min_train + 1:
        return []

    test_size = max(1, row_count // (folds + 1))
    out = []
    train_end = min_train

    while train_end + test_size <= row_count and len(out) < folds:
        out.append((slice(0, train_end), slice(train_end, train_end + test_size)))
        train_end += test_size

    return out

In [ ]:
def post_points(data: pd.DataFrame, target_col: str, folds: int = 4) -> list[tuple[slice, slice]]:
    row_count = len(data)
    min_train = min_train_rows(target_col, "post")

    if row_count <= min_train + 1:
        return []

    test_size = max(1, (row_count - min_train) // folds)

    if test_size == 0:
        test_size = 1

    out = []
    train_end = min_train

    while train_end < row_count:
        test_end = min(row_count, train_end + test_size)
        out.append((slice(0, train_end), slice(train_end, test_end)))
        train_end = test_end

    return out

## Available Sample Sizes by Target

In [ ]:
def sample_row(data: pd.DataFrame, target_col: str) -> pd.DataFrame:
    sample, feat_cols, lag_col = base_block(data, target_col)

    pre_data = sample.loc[sample["pre_regime"].eq(1)].copy()
    post_data = sample.loc[sample["post_regime"].eq(1)].copy()

    return pd.DataFrame(
        {
            "target": [target_name(target_col)],
            "target_col": [target_col],
            "horizon": [horizon(target_col)],
            "rows": [len(sample)],
            "pre_rows": [len(pre_data)],
            "post_rows": [len(post_data)],
            "pre_start": [pre_data["date"].min()],
            "pre_end": [pre_data["date"].max()],
            "post_start": [post_data["date"].min()],
            "post_end": [post_data["date"].max()],
            "feature_count": [len(feat_cols)],
        }
    )

In [ ]:
sample_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data
    sample_parts.append(sample_row(base_data, target_col))

sample_data = pd.concat(sample_parts, ignore_index=True)
display(sample_data.sort_values(["target", "horizon"]).reset_index(drop=True))

## Score Individual Forecasts Across Models and Regimes

In [ ]:
def score_row(
    target_col: str,
    model_name: str,
    regime_name: str,
    actual: np.ndarray,
    pred: np.ndarray,
    base_pred: np.ndarray,
    train_end: pd.Timestamp,
    test_start: pd.Timestamp,
    test_end: pd.Timestamp,
    fold_id: int,
) -> dict:
    return {
        "target": target_name(target_col),
        "target_col": target_col,
        "horizon": horizon(target_col),
        "model": model_name,
        "regime": regime_name,
        "rmse": rmse_value(actual, pred),
        "mae": mae_value(actual, pred),
        "oos_r2": oos_r2(actual, pred, base_pred),
        "row_count": len(actual),
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end,
        "fold_id": fold_id,
    }

In [ ]:
def regime_run(data: pd.DataFrame, target_col: str, regime_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    base_data, feat_cols, lag_col = base_block(data, target_col)
    ar_data, ar_cols, ar_col = ar_block(data, target_col)

    if regime_name == "pre":
        base_data = base_data.loc[base_data["pre_regime"].eq(1)].copy().reset_index(drop=True)
        ar_data = ar_data.loc[ar_data["pre_regime"].eq(1)].copy().reset_index(drop=True)
        points = pre_points(base_data, target_col)

    elif regime_name == "post":
        base_data = base_data.loc[base_data["post_regime"].eq(1)].copy().reset_index(drop=True)
        ar_data = ar_data.loc[ar_data["post_regime"].eq(1)].copy().reset_index(drop=True)
        points = post_points(base_data, target_col)

        if not points and len(base_data) >= 6:
            split_at = max(4, len(base_data) // 2)
            if split_at < len(base_data):
                points = [(slice(0, split_at), slice(split_at, len(base_data)))]

    else:
        raise ValueError("Regime is not supported.")

    if not points:
        return pd.DataFrame(), pd.DataFrame()

    row_list = []
    pred_list = []

    for fold_id, (train_slice, test_slice) in enumerate(points, start=1):
        train_base = base_data.iloc[train_slice].copy()
        test_base = base_data.iloc[test_slice].copy()
        train_ar = ar_data.iloc[train_slice].copy()
        test_ar = ar_data.iloc[test_slice].copy()

        if train_base.empty or test_base.empty:
            continue

        train_feat = train_base[feat_cols].copy()
        train_target = train_base[target_col].copy()
        test_feat = test_base[feat_cols].copy()
        test_target = test_base[target_col].copy()

        base_pred = fit_persistence(train_target, test_feat)

        for model_name in model_list:
            if model_name == "AR":
                if train_ar.empty or test_ar.empty:
                    continue

                pred = model_pred(
                    model_name,
                    train_ar[ar_cols].copy(),
                    train_ar[target_col].copy(),
                    test_ar[ar_cols].copy(),
                    ar_col,
                )

                actual = np.asarray(test_ar[target_col])
                bench = fit_persistence(train_ar[target_col], test_ar[ar_cols].copy())
                train_end = train_ar["date"].max()
                test_start = test_ar["date"].min()
                test_end = test_ar["date"].max()

                row_list.append(
                    score_row(
                        target_col=target_col,
                        model_name=model_name,
                        regime_name=regime_name,
                        actual=actual,
                        pred=np.asarray(pred),
                        base_pred=np.asarray(bench),
                        train_end=train_end,
                        test_start=test_start,
                        test_end=test_end,
                        fold_id=fold_id,
                    )
                )

                pred_list.append(
                    pd.DataFrame(
                        {
                            "date": test_ar["date"].values,
                            "actual": actual,
                            "prediction": np.asarray(pred),
                            "model": model_name,
                            "regime": regime_name,
                            "target": target_name(target_col),
                            "target_col": target_col,
                            "horizon": horizon(target_col),
                            "fold_id": fold_id,
                        }
                    )
                )

            else:
                pred = model_pred(model_name, train_feat, train_target, test_feat, lag_col)

                row_list.append(
                    score_row(
                        target_col=target_col,
                        model_name=model_name,
                        regime_name=regime_name,
                        actual=np.asarray(test_target),
                        pred=np.asarray(pred),
                        base_pred=np.asarray(base_pred),
                        train_end=train_base["date"].max(),
                        test_start=test_base["date"].min(),
                        test_end=test_base["date"].max(),
                        fold_id=fold_id,
                    )
                )

                pred_list.append(
                    pd.DataFrame(
                        {
                            "date": test_base["date"].values,
                            "actual": np.asarray(test_target),
                            "prediction": np.asarray(pred),
                            "model": model_name,
                            "regime": regime_name,
                            "target": target_name(target_col),
                            "target_col": target_col,
                            "horizon": horizon(target_col),
                            "fold_id": fold_id,
                        }
                    )
                )

    score_data = pd.DataFrame(row_list)
    pred_data = pd.concat(pred_list, ignore_index=True) if pred_list else pd.DataFrame()

    return score_data, pred_data

In [ ]:
regime_score_parts = []
regime_pred_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data

    for regime_name in ["pre", "post"]:
        score_data, pred_data = regime_run(base_data, target_col, regime_name)

        if not score_data.empty:
            regime_score_parts.append(score_data)

        if not pred_data.empty:
            regime_pred_parts.append(pred_data)

regime_scores = pd.concat(regime_score_parts, ignore_index=True) if regime_score_parts else pd.DataFrame()
regime_preds = pd.concat(regime_pred_parts, ignore_index=True) if regime_pred_parts else pd.DataFrame()

display(regime_scores.head())
display(regime_preds.head())

## Aggregate Forecast Metrics Within Each Regime

In [ ]:
metric_data = (
    regime_scores.groupby(
        ["target", "target_col", "horizon", "model", "regime"],
        as_index=False,
    )
    .agg(
        rmse=("rmse", "mean"),
        mae=("mae", "mean"),
        oos_r2=("oos_r2", "mean"),
        row_count=("row_count", "sum"),
        fold_count=("fold_id", "nunique"),
    )
    .sort_values(["regime", "target", "horizon", "rmse", "mae"])
    .reset_index(drop=True)
) if not regime_scores.empty else pd.DataFrame()

display(metric_data.round(show_dec))

## Identify the Best Models in Each Regime

In [ ]:
pre_best = (
    metric_data.loc[metric_data["regime"].eq("pre")]
    .sort_values(["target_col", "rmse", "mae"])
    .groupby(["target_col"], as_index=False)
    .first()
    .sort_values(["target", "horizon"])
    .reset_index(drop=True)
) if not metric_data.empty else pd.DataFrame()

In [ ]:
post_best = (
    metric_data.loc[metric_data["regime"].eq("post")]
    .sort_values(["target_col", "rmse", "mae"])
    .groupby(["target_col"], as_index=False)
    .first()
    .sort_values(["target", "horizon"])
    .reset_index(drop=True)
) if not metric_data.empty else pd.DataFrame()

In [ ]:
display(pre_best.round(show_dec))
display(post_best.round(show_dec))

In [ ]:
direct_compare = pre_best[
    ["target", "target_col", "horizon", "model", "rmse", "mae", "oos_r2"]
].rename(
    columns={
        "model": "pre_model",
        "rmse": "pre_rmse",
        "mae": "pre_mae",
        "oos_r2": "pre_oos_r2",
    }
).merge(
    post_best[
        ["target_col", "model", "rmse", "mae", "oos_r2"]
    ].rename(
        columns={
            "model": "post_model",
            "rmse": "post_rmse",
            "mae": "post_mae",
            "oos_r2": "post_oos_r2",
        }
    ),
    on="target_col",
    how="outer",
)

direct_compare["rmse_change_pct"] = 100.0 * (direct_compare["post_rmse"] / direct_compare["pre_rmse"] - 1.0)
direct_compare["mae_change_pct"] = 100.0 * (direct_compare["post_mae"] / direct_compare["pre_mae"] - 1.0)

display(direct_compare.sort_values(["target", "horizon"]).reset_index(drop=True).round(show_dec))

## Test Whether Models Transfer Across Regimes

In [ ]:
def transfer_run(data: pd.DataFrame, target_col: str) -> pd.DataFrame:
    base_data, feat_cols, lag_col = base_block(data, target_col)
    ar_data, ar_cols, ar_col = ar_block(data, target_col)

    train_base = base_data.loc[base_data["pre_regime"].eq(1)].copy()
    test_base = base_data.loc[base_data["post_regime"].eq(1)].copy()
    train_ar = ar_data.loc[ar_data["pre_regime"].eq(1)].copy()
    test_ar = ar_data.loc[ar_data["post_regime"].eq(1)].copy()

    if train_base.empty or test_base.empty:
        return pd.DataFrame()

    row_list = []

    for model_name in model_list:
        if model_name == "AR":
            if train_ar.empty or test_ar.empty:
                continue

            pred = model_pred(
                model_name,
                train_ar[ar_cols].copy(),
                train_ar[target_col].copy(),
                test_ar[ar_cols].copy(),
                ar_col,
            )

            actual = np.asarray(test_ar[target_col])
            bench = fit_persistence(train_ar[target_col], test_ar[ar_cols].copy())
            train_end = train_ar["date"].max()
            test_start = test_ar["date"].min()
            test_end = test_ar["date"].max()

        else:
            pred = model_pred(
                model_name,
                train_base[feat_cols].copy(),
                train_base[target_col].copy(),
                test_base[feat_cols].copy(),
                lag_col,
            )

            actual = np.asarray(test_base[target_col])
            bench = fit_persistence(train_base[target_col], test_base[feat_cols].copy())
            train_end = train_base["date"].max()
            test_start = test_base["date"].min()
            test_end = test_base["date"].max()

        row_list.append(
            score_row(
                target_col=target_col,
                model_name=model_name,
                regime_name="transfer",
                actual=actual,
                pred=np.asarray(pred),
                base_pred=np.asarray(bench),
                train_end=train_end,
                test_start=test_start,
                test_end=test_end,
                fold_id=1,
            )
        )

    return pd.DataFrame(row_list)

In [ ]:
transfer_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data
    part = transfer_run(base_data, target_col)

    if not part.empty:
        transfer_parts.append(part)

transfer_data = pd.concat(transfer_parts, ignore_index=True) if transfer_parts else pd.DataFrame()

display(transfer_data.sort_values(["target", "horizon", "rmse", "mae"]).reset_index(drop=True).round(show_dec))

## Compare Baseline, Linear, and Machine Learning Model Families

In [ ]:
group_data = metric_data.copy()

group_data["group"] = "baseline"
group_data.loc[group_data["model"].isin(linear_list), "group"] = "linear"
group_data.loc[group_data["model"].isin(tree_list), "group"] = "machine_learning"

group_summary = (
    group_data.groupby(["regime", "group"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_mae=("mae", "mean"),
        mean_oos_r2=("oos_r2", "mean"),
    )
    .sort_values(["regime", "group"])
    .reset_index(drop=True)
) if not group_data.empty else pd.DataFrame()

In [ ]:
linear_ml_compare = group_summary.pivot(index="regime", columns="group", values="mean_rmse").reset_index()

if not linear_ml_compare.empty and {"linear", "machine_learning"}.issubset(linear_ml_compare.columns):
    linear_ml_compare["ml_vs_linear_pct"] = 100.0 * (
        linear_ml_compare["machine_learning"] / linear_ml_compare["linear"] - 1.0
    )

display(group_summary.round(show_dec))
display(linear_ml_compare.round(show_dec))

## Evaluate Rolling Forecast Stability Within the Post-COVID Period

In [ ]:
rolling_data = (
    regime_scores.loc[regime_scores["regime"].eq("post")]
    .groupby(["target", "target_col", "horizon", "model", "test_end"], as_index=False)
    .agg(
        rmse=("rmse", "mean"),
        mae=("mae", "mean"),
        oos_r2=("oos_r2", "mean"),
        fold_count=("fold_id", "nunique"),
    )
    .sort_values(["target", "horizon", "model", "test_end"])
    .reset_index(drop=True)
) if not regime_scores.empty else pd.DataFrame()

In [ ]:
rolling_cv = (
    rolling_data.groupby(["target", "target_col", "horizon", "model"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        min_rmse=("rmse", "min"),
        max_rmse=("rmse", "max"),
        row_count=("rmse", "count"),
    )
) if not rolling_data.empty else pd.DataFrame()

if not rolling_cv.empty:
    rolling_cv["cv_rmse"] = rolling_cv["std_rmse"] / rolling_cv["mean_rmse"]

display(rolling_data.round(show_dec))
display(rolling_cv.round(show_dec))

## Measure Coefficient Stability Across Regimes

In [ ]:
def coefficient_row(data: pd.DataFrame, target_col: str, regime_name: str) -> pd.DataFrame:
    sample, feat_cols, lag_col = base_block(data, target_col)

    if regime_name == "pre":
        sample = sample.loc[sample["pre_regime"].eq(1)].copy()
    elif regime_name == "post":
        sample = sample.loc[sample["post_regime"].eq(1)].copy()
    else:
        raise ValueError("Regime is not supported.")

    if sample.empty:
        return pd.DataFrame()

    x_data = sm.add_constant(sample[feat_cols], has_constant="add")
    y_data = sample[target_col].copy()

    fit = sm.OLS(y_data, x_data).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": max(1, horizon(target_col))},
    )

    return pd.DataFrame(
        {
            "target": target_name(target_col),
            "target_col": target_col,
            "horizon": horizon(target_col),
            "regime": regime_name,
            "term": fit.params.index,
            "coefficient": fit.params.values,
            "t_value": fit.tvalues.values,
            "p_value": fit.pvalues.values,
            "row_count": len(sample),
        }
    )

In [ ]:
coef_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data

    for regime_name in ["pre", "post"]:
        part = coefficient_row(base_data, target_col, regime_name)

        if not part.empty:
            coef_parts.append(part)

coef_data = pd.concat(coef_parts, ignore_index=True) if coef_parts else pd.DataFrame()

## Isolate the JOLTS Effect Across Regimes

In [ ]:
jolts_coef = coef_data.loc[coef_data["term"].eq("log_jolts_ratio")].copy()
jolts_coef = jolts_coef.sort_values(["target", "horizon", "regime"]).reset_index(drop=True)

In [ ]:
coef_compare = jolts_coef.pivot_table(
    index=["target", "target_col", "horizon"],
    columns="regime",
    values=["coefficient", "t_value", "p_value", "row_count"],
    aggfunc="first",
).reset_index()

coef_compare.columns = [
    "_".join([str(part) for part in col if str(part) != ""]).strip("_")
    for col in coef_compare.columns.to_flat_index()
]

display(jolts_coef.round(show_dec))
display(coef_compare.round(show_dec))

## Estimate Feature Importance in Flexible Models

In [ ]:
def importance_block(data: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, list[str]]:
    out = add_lag(data, target_col)
    lag_col = lag_col_name(target_col)

    feat_cols = [
        "log_jolts_ratio",
        "unemployment_rate",
        "quits_rate",
        lag_col,
        "fed_funds",
        "hy_oas",
        "high_inflation",
        "tight_labor",
        "credit_stress",
        "jolts_x_unemployment",
        "log_jolts_x_quits",
    ]

    if "log_jolts_ratio" in out.columns:
        out["log_jolts_ratio_square"] = out["log_jolts_ratio"] ** 2
        feat_cols.append("log_jolts_ratio_square")

    feat_cols = [col for col in feat_cols if col in out.columns]

    keep_cols = unique_cols(
        [
            "date",
            target_col,
            *feat_cols,
            "pre_regime",
            "post_regime",
        ]
    )

    out = out.loc[:, keep_cols].copy()

    for col in out.columns:
        if col != "date":
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna().reset_index(drop=True)

    return out, feat_cols

In [ ]:
def fit_importance(model_name: str, train_feat: pd.DataFrame, train_target: pd.Series) -> pd.DataFrame:
    if model_name == "RandomForest":
        fit = clone(tree_model_map["RandomForest"])
    elif model_name == "GradientBoosting":
        fit = clone(tree_model_map["GradientBoosting"])
    else:
        raise ValueError("Model is not supported.")

    fit.fit(train_feat, train_target)

    return pd.DataFrame(
        {
            "feature": train_feat.columns,
            "importance": fit.feature_importances_,
            "model": model_name,
        }
    )

In [ ]:
imp_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data
    sample, feat_cols = importance_block(base_data, target_col)

    for regime_name in ["pre", "post"]:
        if regime_name == "pre":
            reg_data = sample.loc[sample["pre_regime"].eq(1)].copy()
        else:
            reg_data = sample.loc[sample["post_regime"].eq(1)].copy()

        if reg_data.empty or len(reg_data) < 12:
            continue

        train_feat = reg_data[feat_cols].copy()
        train_target = reg_data[target_col].copy()

        for model_name in tree_list:
            part = fit_importance(model_name, train_feat, train_target)
            part["target"] = target_name(target_col)
            part["target_col"] = target_col
            part["horizon"] = horizon(target_col)
            part["regime"] = regime_name

            imp_parts.append(part)

In [ ]:
imp_data = pd.concat(imp_parts, ignore_index=True) if imp_parts else pd.DataFrame()

imp_mean = (
    imp_data.groupby(["feature", "regime"], as_index=False)
    .agg(
        importance=("importance", "mean"),
    )
    .sort_values(["regime", "importance"], ascending=[True, False])
    .reset_index(drop=True)
) if not imp_data.empty else pd.DataFrame()

In [ ]:
top_features = (
    imp_mean.groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
    .head(10)["feature"]
    .tolist()
) if not imp_mean.empty else []

In [ ]:
imp_heat = (
    imp_mean.loc[imp_mean["feature"].isin(top_features)]
    .pivot(index="feature", columns="regime", values="importance")
    .fillna(0.0)
    .reindex(top_features)
) if top_features else pd.DataFrame()

In [ ]:
pre_top = (
    imp_mean.loc[imp_mean["regime"].eq("pre")]
    .sort_values("importance", ascending=False)
    .head(5)
    .reset_index(drop=True)
) if not imp_mean.empty else pd.DataFrame()

In [ ]:
post_top = (
    imp_mean.loc[imp_mean["regime"].eq("post")]
    .sort_values("importance", ascending=False)
    .head(5)
    .reset_index(drop=True)
) if not imp_mean.empty else pd.DataFrame()

In [ ]:
imp_compare = pd.concat(
    [
        pre_top.rename(columns={"feature": "pre_feature", "importance": "pre_importance"}),
        post_top.rename(columns={"feature": "post_feature", "importance": "post_importance"}),
    ],
    axis=1,
) if (not pre_top.empty or not post_top.empty) else pd.DataFrame()

In [ ]:
display(imp_mean.round(show_dec))
display(imp_compare.round(show_dec))
display(imp_heat.round(show_dec))

## Score the Reliability of Individual Labor Market Signals

In [ ]:
def signal_score_row(
    target_col: str,
    signal_name: str,
    regime_name: str,
    actual: np.ndarray,
    pred: np.ndarray,
    base_pred: np.ndarray,
) -> dict:
    return {
        "target": target_name(target_col),
        "target_col": target_col,
        "horizon": horizon(target_col),
        "signal": signal_name,
        "regime": regime_name,
        "rmse": rmse_value(actual, pred),
        "mae": mae_value(actual, pred),
        "oos_r2": oos_r2(actual, pred, base_pred),
    }

In [ ]:
def signal_scores(data: pd.DataFrame, target_col: str, signal_name: str) -> pd.DataFrame:
    sample, feat_cols, lag_col, signal_col = signal_block(data, target_col, signal_name)

    pre_data = sample.loc[sample["pre_regime"].eq(1)].copy().reset_index(drop=True)
    post_data = sample.loc[sample["post_regime"].eq(1)].copy().reset_index(drop=True)

    row_list = []

    pre_folds = pre_points(pre_data, target_col)
    post_folds = post_points(post_data, target_col)

    if not post_folds and len(post_data) >= 6:
        split_at = max(4, len(post_data) // 2)
        if split_at < len(post_data):
            post_folds = [(slice(0, split_at), slice(split_at, len(post_data)))]

    for regime_name, reg_data, reg_folds in [
        ("pre", pre_data, pre_folds),
        ("post", post_data, post_folds),
    ]:
        for train_slice, test_slice in reg_folds:
            train_data = reg_data.iloc[train_slice].copy()
            test_data = reg_data.iloc[test_slice].copy()

            if train_data.empty or test_data.empty:
                continue

            train_feat = train_data[feat_cols].copy()
            train_target = train_data[target_col].copy()
            test_feat = test_data[feat_cols].copy()
            test_target = test_data[target_col].copy()

            pred = fit_ols(train_feat, train_target, test_feat)
            base_pred = fit_persistence(train_target, test_feat)

            row_list.append(
                signal_score_row(
                    target_col=target_col,
                    signal_name=signal_name,
                    regime_name=regime_name,
                    actual=np.asarray(test_target),
                    pred=np.asarray(pred),
                    base_pred=np.asarray(base_pred),
                )
            )

    return pd.DataFrame(row_list)

In [ ]:
def signal_coef(data: pd.DataFrame, target_col: str, signal_name: str) -> pd.DataFrame:
    sample, feat_cols, lag_col, signal_col = signal_block(data, target_col, signal_name)

    row_list = []

    for regime_name in ["pre", "post"]:
        if regime_name == "pre":
            reg_data = sample.loc[sample["pre_regime"].eq(1)].copy()
        else:
            reg_data = sample.loc[sample["post_regime"].eq(1)].copy()

        if reg_data.empty:
            continue

        x_data = sm.add_constant(reg_data[feat_cols], has_constant="add")
        y_data = reg_data[target_col].copy()

        fit = sm.OLS(y_data, x_data).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": max(1, horizon(target_col))},
        )

        row_list.append(
            {
                "target": target_name(target_col),
                "target_col": target_col,
                "horizon": horizon(target_col),
                "signal": signal_name,
                "regime": regime_name,
                "term": signal_col,
                "coefficient": float(fit.params.get(signal_col, np.nan)),
                "t_value": float(fit.tvalues.get(signal_col, np.nan)),
                "p_value": float(fit.pvalues.get(signal_col, np.nan)),
            }
        )

    return pd.DataFrame(row_list)


In [ ]:
signal_score_parts = []
signal_coef_parts = []

for target_col in all_targets:
    base_data = wage_data if target_col.startswith("wage_") else price_data

    for signal_name in signal_list:
        score_part = signal_scores(base_data, target_col, signal_name)
        coef_part = signal_coef(base_data, target_col, signal_name)

        if not score_part.empty:
            signal_score_parts.append(score_part)

        if not coef_part.empty:
            signal_coef_parts.append(coef_part)

In [ ]:
signal_score_data = pd.concat(signal_score_parts, ignore_index=True) if signal_score_parts else pd.DataFrame()
signal_coef_data = pd.concat(signal_coef_parts, ignore_index=True) if signal_coef_parts else pd.DataFrame()

In [ ]:
signal_metric = (
    signal_score_data.groupby(["target", "target_col", "horizon", "signal", "regime"], as_index=False)
    .agg(
        rmse=("rmse", "mean"),
        mae=("mae", "mean"),
        oos_r2=("oos_r2", "mean"),
    )
    .sort_values(["target", "horizon", "signal", "regime"])
    .reset_index(drop=True)
) if not signal_score_data.empty else pd.DataFrame()

In [ ]:
signal_coef_wide = signal_coef_data.pivot_table(
    index=["target", "target_col", "horizon", "signal"],
    columns="regime",
    values=["coefficient", "t_value", "p_value"],
    aggfunc="first",
).reset_index() if not signal_coef_data.empty else pd.DataFrame()

if not signal_coef_wide.empty:
    signal_coef_wide.columns = [
        "_".join([str(part) for part in col if str(part) != ""]).strip("_")
        for col in signal_coef_wide.columns.to_flat_index()
    ]

In [ ]:
signal_compare = signal_metric.pivot_table(
    index=["target", "target_col", "horizon", "signal"],
    columns="regime",
    values=["rmse", "mae", "oos_r2"],
    aggfunc="first",
).reset_index() if not signal_metric.empty else pd.DataFrame()

if not signal_compare.empty:
    signal_compare.columns = [
        "_".join([str(part) for part in col if str(part) != ""]).strip("_")
        for col in signal_compare.columns.to_flat_index()
    ]

In [ ]:
reliability_data = signal_compare.merge(
    signal_coef_wide,
    on=["target", "target_col", "horizon", "signal"],
    how="left",
) if (not signal_compare.empty and not signal_coef_wide.empty) else pd.DataFrame()

In [ ]:
display(signal_metric.round(show_dec))
display(signal_coef_data.round(show_dec))
display(reliability_data.round(show_dec))

In [ ]:
if not reliability_data.empty:
    reliability_data["rmse_change_pct"] = 100.0 * (
        reliability_data["rmse_post"] / reliability_data["rmse_pre"] - 1.0
    )

    reliability_data["sign_same"] = (
        np.sign(reliability_data["coefficient_pre"]).fillna(0.0)
        == np.sign(reliability_data["coefficient_post"]).fillna(0.0)
    )

    reliability_data["coef_ratio"] = np.abs(
        reliability_data["coefficient_post"] / reliability_data["coefficient_pre"].replace(0.0, np.nan)
    )

    reliability_data["reliability"] = "Unreliable"

    reliable_mask = (
        reliability_data["rmse_change_pct"].le(15.0)
        & reliability_data["sign_same"]
        & reliability_data["coef_ratio"].between(0.50, 1.50, inclusive="both")
    )

    partial_mask = (
        reliability_data["rmse_change_pct"].gt(15.0)
        & reliability_data["rmse_change_pct"].le(30.0)
        & reliability_data["sign_same"]
    )

    reliability_data.loc[reliable_mask, "reliability"] = "Reliable"
    reliability_data.loc[partial_mask, "reliability"] = "Partial"

    reliability_data["flag"] = 0
    reliability_data.loc[reliability_data["reliability"].eq("Partial"), "flag"] = 1
    reliability_data.loc[reliability_data["reliability"].eq("Reliable"), "flag"] = 2

In [ ]:
reliability_matrix = reliability_data.pivot(
    index="target_col",
    columns="signal",
    values="flag",
) if not reliability_data.empty else pd.DataFrame()

In [ ]:
reliability_text = reliability_data.pivot(
    index="target_col",
    columns="signal",
    values="reliability",
) if not reliability_data.empty else pd.DataFrame()

In [ ]:
display(reliability_data.sort_values(["target", "horizon", "signal"]).reset_index(drop=True).round(show_dec))
display(reliability_text)

## Build Final Comparison Tables Across Regimes

In [ ]:
model_compare = metric_data.pivot_table(
    index=["target", "target_col", "horizon", "model"],
    columns="regime",
    values=["rmse", "mae", "oos_r2"],
    aggfunc="first",
).reset_index() if not metric_data.empty else pd.DataFrame()

if not model_compare.empty:
    model_compare.columns = [
        "_".join([str(part) for part in col if str(part) != ""]).strip("_")
        for col in model_compare.columns.to_flat_index()
    ]

    model_compare["rmse_change_pct"] = 100.0 * (
        model_compare["rmse_post"] / model_compare["rmse_pre"] - 1.0
    )
    model_compare["mae_change_pct"] = 100.0 * (
        model_compare["mae_post"] / model_compare["mae_pre"] - 1.0
    )

In [ ]:
model_change = (
    model_compare.groupby(["model"], as_index=False)
    .agg(
        mean_pre_rmse=("rmse_pre", "mean"),
        mean_post_rmse=("rmse_post", "mean"),
        mean_rmse_change_pct=("rmse_change_pct", "mean"),
        mean_pre_mae=("mae_pre", "mean"),
        mean_post_mae=("mae_post", "mean"),
        mean_mae_change_pct=("mae_change_pct", "mean"),
    )
    .sort_values("mean_rmse_change_pct")
    .reset_index(drop=True)
) if not model_compare.empty else pd.DataFrame()

display(model_compare.round(show_dec))
display(model_change.round(show_dec))

## Visualize Forecast Performance Before and After COVID-19

In [ ]:
plot_data = metric_data.copy()
plot_data["regime_label"] = plot_data["regime"].map({"pre": "Pre-2020", "post": "Post-2021"})

fig, axes = plt.subplots(3, 3, figsize=(22, 15), sharey=False)
axes = axes.ravel()

for ax, target_col in zip(axes, all_targets):
    part = plot_data.loc[plot_data["target_col"].eq(target_col)].copy()
    part = part.sort_values(["regime_label", "rmse", "model"]).reset_index(drop=True)

    sns.barplot(
        data=part,
        x="model",
        y="rmse",
        hue="regime_label",
        ax=ax,
    )

    ax.set_title(target_label(target_col), fontsize=13, pad=10, weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("RMSE", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, axis="y", alpha=0.20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if ax.get_legend() is not None:
        ax.get_legend().set_title("")
        ax.get_legend().set_frame_on(False)

for ax in axes[len(all_targets):]:
    ax.set_axis_off()

fig.suptitle(
    "Model performance by regime and target",
    fontsize=18,
    weight="bold",
    y=1.02,
)
fig.tight_layout()
fig.savefig(fig_root / "results_and_audit_model_comparison.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

In [ ]:
scatter_data = (
    metric_data.groupby(["model", "regime"], as_index=False)
    .agg(
        rmse=("rmse", "mean"),
    )
    .pivot(index="model", columns="regime", values="rmse")
    .reset_index()
)

scatter_data["group"] = "Baseline"
scatter_data.loc[scatter_data["model"].isin(linear_list), "group"] = "Linear"
scatter_data.loc[scatter_data["model"].isin(tree_list), "group"] = "Machine learning"

color_map = {
    "Baseline": "#6c757d",
    "Linear": "#1f77b4",
    "Machine learning": "#d62728",
}

fig, ax = plt.subplots(figsize=(10, 8))

for _, row in scatter_data.iterrows():
    ax.scatter(
        row["pre"],
        row["post"],
        s=140,
        color=color_map[row["group"]],
        alpha=0.90,
    )
    ax.text(
        row["pre"],
        row["post"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
    )

low = min(scatter_data["pre"].min(), scatter_data["post"].min())
high = max(scatter_data["pre"].max(), scatter_data["post"].max())

pad = 0.03 * (high - low)
ax.plot(
    [low - pad, high + pad],
    [low - pad, high + pad],
    linestyle="--",
    linewidth=1.4,
    color="black",
    alpha=0.7,
)

ax.set_xlim(low - pad, high + pad)
ax.set_ylim(low - pad, high + pad)
ax.set_title("Pre-2020 versus post-2021 RMSE by model", fontsize=16, pad=12, weight="bold")
ax.set_xlabel("Pre-2020 average RMSE", fontsize=12)
ax.set_ylabel("Post-2021 average RMSE", fontsize=12)
ax.grid(True, alpha=0.20)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

handles = []
labels = []
for group_name, color in color_map.items():
    handle = plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=10)
    handles.append(handle)
    labels.append(group_name)

ax.legend(handles, labels, frameon=False, title="")

fig.tight_layout()
fig.savefig(fig_root / "results_and_audit_linear_vs_ml.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    imp_heat,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Average feature importance"},
    ax=ax,
)

ax.set_title("Feature importance shift across regimes", fontsize=16, pad=12, weight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

fig.tight_layout()
fig.savefig(fig_root / "results_and_audit_feature_shift.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

## Identify the Strongest Linear Models After COVID-19

In [ ]:
best_linear = (
    metric_data.loc[metric_data["regime"].eq("post") & metric_data["model"].isin(linear_list)]
    .groupby("model", as_index=False)
    .agg(rmse=("rmse", "mean"))
    .sort_values("rmse")
    .iloc[0]["model"]
)

In [ ]:
best_tree = (
    metric_data.loc[metric_data["regime"].eq("post") & metric_data["model"].isin(tree_list)]
    .groupby("model", as_index=False)
    .agg(rmse=("rmse", "mean"))
    .sort_values("rmse")
    .iloc[0]["model"]
)

In [ ]:
plot_linear = (
    rolling_data.loc[rolling_data["model"].eq(best_linear)]
    .groupby("test_end", as_index=False)
    .agg(rmse=("rmse", "mean"))
)

plot_tree = (
    rolling_data.loc[rolling_data["model"].eq(best_tree)]
    .groupby("test_end", as_index=False)
    .agg(rmse=("rmse", "mean"))
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    plot_linear["test_end"],
    plot_linear["rmse"],
    marker="o",
    linewidth=2.2,
    label=best_linear,
)

ax.plot(
    plot_tree["test_end"],
    plot_tree["rmse"],
    marker="o",
    linewidth=2.2,
    label=best_tree,
)

ax.set_title("Rolling post-2021 RMSE for the best linear and tree models", fontsize=16, pad=12, weight="bold")
ax.set_xlabel("Test window end date", fontsize=12)
ax.set_ylabel("RMSE", fontsize=12)
ax.grid(True, axis="y", alpha=0.20)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, title="Model")

fig.tight_layout()
fig.savefig(fig_root / "results_and_audit_rolling_rmse.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.heatmap(
    reliability_matrix,
    annot=reliability_text,
    fmt="",
    cmap=ListedColormap(["#d62728", "#ffdd57", "#2ca02c"]),
    cbar=False,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)

ax.set_title("Post-COVID signal reliability matrix", fontsize=16, pad=12, weight="bold")
ax.set_xlabel("Signal", fontsize=12)
ax.set_ylabel("Target", fontsize=12)
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

fig.tight_layout()
fig.savefig(fig_root / "results_and_audit_reliability_matrix.png", dpi=save_dpi, bbox_inches="tight")
plt.show()

In [ ]:
metric_data.to_csv(report_root / "results_and_audit_metrics.csv", index=False)
pre_best.to_csv(report_root / "results_and_audit_pre_best.csv", index=False)
post_best.to_csv(report_root / "results_and_audit_post_best.csv", index=False)
direct_compare.to_csv(report_root / "results_and_audit_direct_compare.csv", index=False)
transfer_data.to_csv(report_root / "results_and_audit_transfer.csv", index=False)
group_summary.to_csv(report_root / "results_and_audit_group_summary.csv", index=False)
linear_ml_compare.to_csv(report_root / "results_and_audit_linear_ml_compare.csv", index=False)
rolling_data.to_csv(report_root / "results_and_audit_rolling_data.csv", index=False)
rolling_cv.to_csv(report_root / "results_and_audit_rolling_cv.csv", index=False)
coef_data.to_csv(report_root / "results_and_audit_coefficients.csv", index=False)
coef_compare.to_csv(report_root / "results_and_audit_jolts_coefficients.csv", index=False)
imp_mean.to_csv(report_root / "results_and_audit_feature_importance.csv", index=False)
imp_compare.to_csv(report_root / "results_and_audit_feature_top_compare.csv", index=False)
signal_metric.to_csv(report_root / "results_and_audit_signal_metrics.csv", index=False)
signal_coef_data.to_csv(report_root / "results_and_audit_signal_coefficients.csv", index=False)
reliability_data.to_csv(report_root / "results_and_audit_reliability.csv", index=False)
model_compare.to_csv(report_root / "results_and_audit_model_compare.csv", index=False)
model_change.to_csv(report_root / "results_and_audit_model_change.csv", index=False)
sample_data.to_csv(report_root / "results_and_audit_sample_data.csv", index=False)
regime_scores.to_csv(report_root / "results_and_audit_regime_scores.csv", index=False)
regime_preds.to_csv(model_root / "results_and_audit_regime_predictions.csv", index=False)

In [ ]:
display(metric_data.round(show_dec))
display(pre_best.round(show_dec))
display(post_best.round(show_dec))
display(direct_compare.round(show_dec))
display(group_summary.round(show_dec))
display(linear_ml_compare.round(show_dec))
display(coef_compare.round(show_dec))
display(imp_compare.round(show_dec))
display(reliability_data.round(show_dec))
display(model_change.round(show_dec))
display(sample_data.round(show_dec))
display(rolling_cv.round(show_dec))

## Conclusion